In [1]:
# FINAL MODEL - Copy this to our thesis Jupyter
import pandas as pd
import numpy as np

df = pd.read_csv('DillibabuSarva_DefectDataset.csv')

# BEST 10 FEATURES (from feature selection)
FEATURES = ['nosi','dit','cbo','rfc','maxNestedBlocks',
            'uniqueWordsQty','assignmentsQty','numbersQty',
            'tryCatchQty','parenthesizedExpsQty']

X = df[FEATURES]
y = df['defect']

# --- For Colab with sklearn (RECOMMENDED) ---
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

# Final Model: Random Forest (Better than Logistic)
final_model = AdaBoostClassifier(estimator=None, n_estimators=50, learning_rate=1.0, algorithm='SAMME', random_state=None)
final_model.fit(X_train, y_train)

# Evaluation
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:,1]

print(f"Test Accuracy: { (y_test==y_pred).mean():.3f}")
print(f"Test AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(classification_report(y_test, y_pred))

# Feature importance for thesis table
importance = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': final_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(importance)

# Save final model
import pickle
with open('final_defect_model.pkl','wb') as f:
    pickle.dump(final_model, f)

Test Accuracy: 0.760
Test AUC: 0.817
              precision    recall  f1-score   support

           0       0.83      0.65      0.73       908
           1       0.71      0.87      0.78       908

    accuracy                           0.76      1816
   macro avg       0.77      0.76      0.76      1816
weighted avg       0.77      0.76      0.76      1816

                Feature  Importance
0                  nosi    0.400632
2                   cbo    0.188271
1                   dit    0.155798
3                   rfc    0.140500
7            numbersQty    0.055297
8           tryCatchQty    0.038129
5        uniqueWordsQty    0.021373
4       maxNestedBlocks    0.000000
6        assignmentsQty    0.000000
9  parenthesizedExpsQty    0.000000


In [2]:
import joblib
joblib.dump(final_model, 'AdaBoost_final_defect_prediction_model.pkl')
print("Saved to AdaBoost_final_defect_prediction_model.pkl")

Saved to AdaBoost_final_defect_prediction_model.pkl


In [3]:
model = joblib.load('AdaBoost_final_defect_prediction_model.pkl') # String path, not variable
new_prediction = model.predict([[10, 5, 20, 70, 5, 200, 60, 30, 3, 10]])
print(f"Defect? {new_prediction[0]}")

Defect? 0


C:\Anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but AdaBoostClassifier was fitted with feature names
  warnings.warn(
